<a href="https://colab.research.google.com/github/Nakib-Nasrullah/Heart_disease/blob/main/Transformer_Based_ECG_Classification.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install wfdb pandas numpy scikit-learn tensorflow matplotlib seaborn
import os
import wfdb
import numpy as np
import pandas as pd
import tensorflow as tf
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score

from tensorflow.keras.layers import *
from tensorflow.keras.models import Model


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 163.9/163.9 kB 6.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.9/10.9 MB 62.1 MB/s eta 0:00:00
  Attempting uninstall: pandas
    Found existing installation: pandas 2.2.2
    Uninstalling pandas-2.2.2:
      Successfully uninstalled pandas-2.2.2
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires pandas==2.2.2, but you have pandas 3.0.2 which is incompatible.
db-dtypes 1.5.1 requires pandas<3.0.0,>=1.5.3, but you have pandas 3.0.2 which is incompatible.
bqplot 0.12.45 requires pandas<3.0.0,>=1.0.0, but you have pandas 3.0.2 which is incompatible.
gradio 5.50.0 requires pandas<3.0,>=1.0, but you have pandas 3.0.2 which is incompatible.


# **Download Dataset**

In [ ]:
DATA_DIR = "mitdb"

if not os.path.exists(DATA_DIR):
    wfdb.dl_database('mitdb', dl_dir=DATA_DIR)

print("Dataset ready!")

Generating record list for: 100
Generating record list for: 101
Generating record list for: 102
Generating record list for: 103
Generating record list for: 104
Generating record list for: 105
Generating record list for: 106
Generating record list for: 107
Generating record list for: 108
Generating record list for: 109
Generating record list for: 111
Generating record list for: 112
Generating record list for: 113
Generating record list for: 114
Generating record list for: 115
Generating record list for: 116
Generating record list for: 117
Generating record list for: 118
Generating record list for: 119
Generating record list for: 121
Generating record list for: 122
Generating record list for: 123
Generating record list for: 124
Generating record list for: 200
Generating record list for: 201
Generating record list for: 202
Generating record list for: 203
Generating record list for: 205
Generating record list for: 207
Generating record list for: 208
Generating record list for: 209
Generati

# **Beat Extraction**

In [ ]:
WINDOW = 187
HALF = WINDOW // 2

label_map = {
    'N': 0, 'L': 0, 'R': 0, 'e': 0, 'j': 0,
    'A': 1, 'a': 1, 'J': 1, 'S': 1,
    'V': 2, 'E': 2,
    'F': 3
}

beats = []

records = sorted([f.split('.')[0] for f in os.listdir(DATA_DIR) if f.endswith('.dat')])

for record in records:
    try:
        signal, _ = wfdb.rdsamp(os.path.join(DATA_DIR, record))
        ann = wfdb.rdann(os.path.join(DATA_DIR, record), 'atr')
    except:
        continue

    ecg = signal[:, 0]

    for r, sym in zip(ann.sample, ann.symbol):
        if sym not in label_map:
            continue

        if r - HALF < 0 or r + HALF >= len(ecg):
            continue

        beat = ecg[r-HALF:r+HALF+1]
        beats.append([record] + beat.tolist() + [label_map[sym]])

columns = ["record_id"] + [f"f{i}" for i in range(WINDOW)] + ["label"]
df = pd.DataFrame(beats, columns=columns)

df.to_csv("mitbih_patient_level.csv", index=False)
print("Dataset created:", df.shape)


Dataset created: (101426, 189)


# **Patient-Level Split**

In [ ]:
df = pd.read_csv("mitbih_patient_level.csv")

patients = df['record_id'].unique()

trainval_patients, test_patients = train_test_split(
    patients, test_size=0.20, random_state=42
)

train_patients, val_patients = train_test_split(
    trainval_patients, test_size=0.10, random_state=42
)

train_df = df[df['record_id'].isin(train_patients)]
val_df   = df[df['record_id'].isin(val_patients)]
test_df  = df[df['record_id'].isin(test_patients)]

print("No patient overlap ✔")


No patient overlap ✔


# **Data Preparation**

In [ ]:
def prepare(df):
    X = df.iloc[:, 1:-1].values
    y = df['label'].values.astype(int)

    X = (X - X.mean(axis=1, keepdims=True)) / \
        (X.std(axis=1, keepdims=True) + 1e-8)

    return X.reshape(-1, 187, 1), y

X_train, y_train = prepare(train_df)
X_val, y_val     = prepare(val_df)
X_test, y_test   = prepare(test_df)

print(X_train.shape, X_val.shape, X_test.shape)


(72737, 187, 1) (8245, 187, 1) (20444, 187, 1)


# **Class Weights**

In [ ]:
weights = compute_class_weight(
    class_weight='balanced',
    classes=np.unique(y_train),
    y=y_train
)

class_weights = dict(enumerate(weights))
print(class_weights)


{0: np.float64(0.27984379809172055), 1: np.float64(8.1616921005386), 2: np.float64(3.5543881939014854), 3: np.float64(44.029661016949156)}


# **Focal Loss**

In [ ]:
def focal_loss(gamma=2., alpha=0.25):
    def loss(y_true, y_pred):
        y_true = tf.one_hot(tf.cast(y_true, tf.int32), depth=4)
        ce = tf.keras.losses.categorical_crossentropy(y_true, y_pred)
        pt = tf.reduce_sum(y_true * y_pred, axis=-1)
        return alpha * tf.pow(1. - pt, gamma) * ce
    return loss


# **Positional Encoding**

In [ ]:
class PositionalEncoding(Layer):
    def __init__(self, seq_len, d_model):
        super().__init__()
        self.pos_encoding = self.positional_encoding(seq_len, d_model)

    def get_angles(self, pos, i, d_model):
        return pos / tf.pow(10000., (2 * (i // 2)) / tf.cast(d_model, tf.float32))

    def positional_encoding(self, length, d_model):
        pos = tf.range(length, dtype=tf.float32)[:, tf.newaxis]
        i = tf.range(d_model, dtype=tf.float32)[tf.newaxis, :]

        angles = self.get_angles(pos, i, d_model)

        sines = tf.sin(angles[:, 0::2])
        cosines = tf.cos(angles[:, 1::2])

        return tf.concat([sines, cosines], axis=-1)[tf.newaxis, ...]

    def call(self, x):
        return x + self.pos_encoding[:, :tf.shape(x)[1], :]


# **Transformer Encoder Block**

In [ ]:
def transformer_block(x, head_size=64, num_heads=4, ff_dim=128, dropout=0.1):

    attn = MultiHeadAttention(num_heads=num_heads, key_dim=head_size)(x, x)
    attn = Dropout(dropout)(attn)
    x = LayerNormalization(epsilon=1e-6)(x + attn)

    ff = Dense(ff_dim, activation='relu')(x)
    ff = Dense(x.shape[-1])(ff)
    ff = Dropout(dropout)(ff)

    return LayerNormalization(epsilon=1e-6)(x + ff)


# **Build Transformer Model**

In [ ]:
"""def build_transformer():

    inputs = Input(shape=(187,1))

    # Project to higher dimension
    x = Dense(64)(inputs)

    # Add positional encoding
    x = PositionalEncoding(187, 64)(x)

    # Transformer layers
    for _ in range(4):
        x = transformer_block(x)

    # Global pooling
    x = GlobalAveragePooling1D()(x)

    x = Dense(128, activation='relu')(x)
    x = Dropout(0.5)(x)

    outputs = Dense(4, activation='softmax')(x)

    return Model(inputs, outputs)

model = build_transformer()
model.summary()"""


Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer         │ (None, 187, 1)    │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense (Dense)       │ (None, 187, 64)   │        128 │ input_layer[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ positional_encoding │ (None, 187, 64)   │          0 │ dense[0][0]       │
│ (PositionalEncodin… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ multi_head_attenti… │ (None, 187, 64)   │     66,368 │ positional_encod… │
│ (MultiHeadAttentio… │                   │            │ positional_encod… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_1 (Dropout) │ (None, 187, 64)   │          0 │ multi_head_atten… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add (Add)           │ (None, 187, 64)   │          0 │ positional_encod… │
│                     │                   │            │ dropout_1[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ layer_normalization │ (None, 187, 64)   │        128 │ add[0][0]         │
│ (LayerNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_1 (Dense)     │ (None, 187, 128)  │      8,320 │ layer_normalizat… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_2 (Dense)     │ (None, 187, 64)   │      8,256 │ dense_1[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_2 (Dropout) │ (None, 187, 64)   │          0 │ dense_2[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add_1 (Add)         │ (None, 187, 64)   │          0 │ layer_normalizat… │
│                     │                   │            │ dropout_2[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ layer_normalizatio… │ (None, 187, 64)   │        128 │ add_1[0][0]       │
│ (LayerNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ multi_head_attenti… │ (None, 187, 64)   │     66,368 │ layer_normalizat… │
│ (MultiHeadAttentio… │                   │            │ layer_normalizat… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_4 (Dropout) │ (None, 187, 64)   │          0 │ multi_head_atten… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add_2 (Add)         │ (None, 187, 64)   │          0 │ layer_normalizat… │
│                     │                   │            │ dropout_4[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ layer_normalizatio… │ (None, 187, 64)   │        128 │ add_2[0][0]       │
│ (LayerNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_3 (Dense)     │ (None, 187, 128)  │      8,320 │ layer_normalizat… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_4 (Dense)     │ (None, 187, 64)   │      8,256 │ dense_3[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_5 (Dropout) │ (None, 187, 64)   │          0 │ dense_4[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add_3 (Add)         │ (None, 187, 64)   │          0 │ layer_normalizat

 Total params: 341,764 (1.30 MB)

 Trainable params: 341,764 (1.30 MB)

 Non-trainable params: 0 (0.00 B)

# **Compile**

In [17]:
"""model.compile(
    optimizer=tf.keras.optimizers.Adam(0.0003),
    loss=focal_loss(),
    metrics=['accuracy']
)"""


# **Train**

In [16]:
"""history = model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=20,
    batch_size=32,
    class_weight=class_weights,
    verbose=1
)"""

Epoch 1/20
 665/2274 ━━━━━━━━━━━━━━━━━━━━ 34:16 1s/step - accuracy: 0.3926 - loss: 0.1076

KeyboardInterrupt: 

# **Save Model**

In [ ]:
"""model.save("ecg_transformer.keras")"""

# **Evaluation**

In [ ]:
"""y_pred_probs = model.predict(X_test)
y_pred = y_pred_probs.argmax(axis=1)

print(classification_report(y_test, y_pred))"""

# **Confusion Matrix**

In [ ]:
"""cm = confusion_matrix(y_test, y_pred)

plt.figure(figsize=(6,5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
plt.title("Confusion Matrix")
plt.show()"""

# **ROC-AUC**

In [ ]:
"""y_test_onehot = tf.keras.utils.to_categorical(y_test, 4)

auc = roc_auc_score(y_test_onehot, y_pred_probs, multi_class='ovr')
print("ROC-AUC:", auc)"""

# **Multi-Scale CNN Block**

In [18]:
def cnn_block(inputs):

    c1 = Conv1D(32, 3, padding='same', activation='relu')(inputs)
    c2 = Conv1D(32, 5, padding='same', activation='relu')(inputs)
    c3 = Conv1D(32, 7, padding='same', activation='relu')(inputs)

    x = Concatenate()([c1, c2, c3])
    x = BatchNormalization()(x)
    x = MaxPooling1D(2)(x)

    return x

# **Build Hybrid Model**

In [20]:
def build_hybrid_model():

    inputs = Input(shape=(187,1))

    # 🔹 CNN Feature Extraction
    x = cnn_block(inputs)
    x = cnn_block(x)
    x = cnn_block(x)

    # 🔹 Project to Transformer space
    x = Conv1D(64, 1)(x)

    # 🔹 Positional Encoding
    seq_len = x.shape[1]
    x = PositionalEncoding(seq_len, 64)(x)

    # 🔹 Transformer Layers
    for _ in range(3):
        x = transformer_block(x)

    # 🔹 Attention Pooling
    attention = Dense(1, activation='tanh')(x)
    attention = Softmax(axis=1)(attention)
    x = Lambda(lambda x_and_attention: tf.reduce_sum(x_and_attention[0] * x_and_attention[1], axis=1))([x, attention])

    # 🔹 Dense Layers
    x = Dense(128, activation='relu')(x)
    x = Dropout(0.5)(x)

    outputs = Dense(4, activation='softmax')(x)

    return Model(inputs, outputs)

model = build_hybrid_model()
model.summary()

Model: "functional_1"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_2       │ (None, 187, 1)    │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d_10 (Conv1D)  │ (None, 187, 32)   │        128 │ input_layer_2[0]… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d_11 (Conv1D)  │ (None, 187, 32)   │        192 │ input_layer_2[0]… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d_12 (Conv1D)  │ (None, 187, 32)   │        256 │ input_layer_2[0]… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ concatenate_3       │ (None, 187, 96)   │          0 │ conv1d_10[0][0],  │
│ (Concatenate)       │                   │            │ conv1d_11[0][0],  │
│                     │                   │            │ conv1d_12[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 187, 96)   │        384 │ concatenate_3[0]… │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling1d_3     │ (None, 93, 96)    │          0 │ batch_normalizat… │
│ (MaxPooling1D)      │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d_13 (Conv1D)  │ (None, 93, 32)    │      9,248 │ max_pooling1d_3[… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d_14 (Conv1D)  │ (None, 93, 32)    │     15,392 │ max_pooling1d_3[… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d_15 (Conv1D)  │ (None, 93, 32)    │     21,536 │ max_pooling1d_3[… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ concatenate_4       │ (None, 93, 96)    │          0 │ conv1d_13[0][0],  │
│ (Concatenate)       │                   │            │ conv1d_14[0][0],  │
│                     │                   │            │ conv1d_15[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 93, 96)    │        384 │ concatenate_4[0]… │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling1d_4     │ (None, 46, 96)    │          0 │ batch_normalizat… │
│ (MaxPooling1D)      │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d_16 (Conv1D)  │ (None, 46, 32)    │      9,248 │ max_pooling1d_4[… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d_17 (Conv1D)  │ (None, 46, 32)    │     15,392 │ max_pooling1d_4[… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d_18 (Conv1D)  │ (None, 46, 32)    │     21,536 │ max_pooling1d_4[… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ concatenate_5       │ (None, 46, 96)    │          0 │ conv1d_16[0][0],  │
│ (Concatenate)       │                   │            │ conv1d_17[0][0],  │
│                     │                   │            │ conv1d_18[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 46, 96)    │        384 │ concatenate_5[0]… │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling1d_5     │ (None, 23, 96)    │          0 │ batch_normalizat

 Total params: 358,789 (1.37 MB)

 Trainable params: 358,213 (1.37 MB)

 Non-trainable params: 576 (2.25 KB)

# **Compile**

In [21]:
model.compile(
    optimizer=tf.keras.optimizers.Adam(0.0003),
    loss=focal_loss(),
    metrics=['accuracy']
)

# **Train**

In [ ]:
history = model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=35,
    batch_size=64,
    class_weight=class_weights,
    verbose=1
)

Epoch 1/35
